[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/01_introduccion_regex.ipynb)

# Capítulo 1. Introducción a la Analítica Textual y limpieza con Regex

Este notebook forma parte del apunte del curso **Analítica Textual** y está preparado para ejecutarse en Google Colab o en Jupyter local.


In [ ]:
# Setup opcional para Colab
# En general, este notebook corre sin instalaciones adicionales.
# Si tu entorno no tiene las librerías base, descomenta la línea siguiente.
# !pip -q install pandas


## Objetivos

Al terminar este capítulo deberías poder:

- describir qué problemas resuelve la analítica textual en organizaciones;
- reconocer desafíos frecuentes del lenguaje natural;
- limpiar texto con expresiones regulares en Python;
- construir una primera rutina reproducible de preprocesamiento.

## 1.1 ¿Qué es la analítica textual?

La analítica textual estudia cómo transformar texto libre en estructuras que permitan describir, comparar, clasificar o generar información útil. El texto es una fuente rica, pero también desordenada: contiene ambigüedad, ruido, abreviaturas, errores ortográficos, ironía, referencias culturales y cambios de contexto.

Hoy aparece en múltiples procesos:

- atención al cliente: clasificación de tickets, resumen de conversaciones, detección de urgencia;
- finanzas: lectura de noticias, reportes y documentos regulatorios;
- salud: apoyo a revisión de fichas clínicas, triage y normalización de notas;
- legaltech: búsqueda semántica, extracción de cláusulas y análisis de contratos;
- operaciones: minería de correos, encuestas abiertas y reportes de incidentes.

## 1.2 Retos del lenguaje natural

Antes de modelar, conviene tener claro qué hace difícil al texto:

1. Variación léxica: dos documentos pueden expresar la misma idea con palabras distintas.
2. Ambigüedad: una palabra cambia de significado según el contexto.
3. Ruido: URLs, etiquetas HTML, emojis, espacios extra, firmas o texto duplicado.
4. Dependencia del dominio: el vocabulario de medicina, derecho o banca no se comporta igual.
5. Cambio temporal: nuevos términos, nombres y eventos modifican la distribución del lenguaje.

Una parte importante del trabajo analítico consiste en reducir ruido sin destruir señal útil.

## 1.3 Expresiones regulares

Las expresiones regulares permiten describir patrones en texto. En Python se usan con el módulo `re`.

Patrones comunes:

- `\d+`: una o más cifras;
- `\w+`: caracteres alfanuméricos;
- `\s+`: espacios, tabulaciones o saltos de línea;
- `https?://\S+`: URL simple;
- `[^a-záéíóúñ\s]`: caracteres que no pertenecen al conjunto permitido.

## 1.4 Primer ejemplo de limpieza

Supongamos que recibimos comentarios desde una plataforma digital:


In [ ]:
comentarios = [
    "Excelente servicio!!! Llegó en 24 hrs. Más info en https://tienda.cl",
    "@soporte no responde... pedido #12345 aún pendiente :(",
    "Cliente indica:   producto con falla     en pantalla.",
    "<p>Necesito devolución</p> urgente!!!"
]


Podemos limpiar el ruido más evidente:


In [ ]:
import re

def limpiar_texto(texto: str) -> str:
    texto = texto.lower()
    texto = re.sub(r"https?://\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"[^a-záéíóúñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

comentarios_limpios = [limpiar_texto(c) for c in comentarios]
comentarios_limpios


Resultado esperado:


In [ ]:
[
    "excelente servicio llegó en 24 hrs más info en",
    "no responde pedido 12345 aún pendiente",
    "cliente indica producto con falla en pantalla",
    "necesito devolución urgente"
]


## 1.5 Extracción de patrones

Regex no solo sirve para limpiar; también permite extraer señales:


In [ ]:
texto = "Contactar a maria@empresa.cl antes del 05-04-2026 por caso 9981."

email = re.findall(r"[\w\.-]+@[\w\.-]+\.\w+", texto)
fechas = re.findall(r"\b\d{2}-\d{2}-\d{4}\b", texto)
ids = re.findall(r"\b\d{4}\b", texto)

print(email)
print(fechas)
print(ids)


Esto es útil para:

- identificar datos personales;
- detectar códigos de seguimiento;
- normalizar fechas;
- separar contenido libre de metadatos.

## 1.6 Buenas prácticas de limpieza

No toda limpieza es buena limpieza. Algunas recomendaciones:

1. Conserva tildes y caracteres relevantes si el idioma importa.
2. No elimines números si contienen señal de negocio.
3. Documenta cada regla y por qué existe.
4. Prueba la rutina sobre ejemplos reales, no solo sobre casos ideales.
5. Mantén una versión del texto original para auditoría.

## 1.7 Mini pipeline reutilizable


In [ ]:
import pandas as pd

df = pd.DataFrame({"texto_original": comentarios})
df["texto_limpio"] = df["texto_original"].apply(limpiar_texto)
df["n_tokens_aprox"] = df["texto_limpio"].str.split().str.len()
df


Con esto ya tenemos una tabla base para las siguientes semanas.

## 1.8 Errores frecuentes

- usar una regla agresiva que borra información útil;
- mezclar limpieza lingüística con decisiones de negocio sin documentarlas;
- asumir que un patrón cubre todos los casos del mundo real;
- olvidar revisar ejemplos manualmente.

## Ejercicios

1. Extiende `limpiar_texto` para eliminar números de pedido del tipo `#12345`.
2. Crea una regex para capturar montos como `$19.990` o `$2500`.
3. Modifica el pipeline para detectar comentarios con la palabra `urgente`.

## Idea clave

La analítica textual comienza mucho antes del modelado. Una limpieza clara, mínima y reproducible suele mejorar más el proyecto que una técnica compleja aplicada sobre datos caóticos.
